<div style="background: linear-gradient(135deg, #2d6a4f 0%, #52b788 40%, #1a759f 100%); padding: 35px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; font-family: 'Segoe UI', sans-serif; font-size: 2.2em; margin: 0; font-weight: 700;">📊 Análisis de Segmentación de Clientes</h1>
  <p style="color: #d8f3dc; font-size: 1.1em; margin: 12px 0 0;">E-Commerce Churn Prediction &nbsp;|&nbsp; No Country — Equipo 40</p>
</div>

> **Objetivo:** Ir más allá de la predicción binaria y construir **segmentos accionables** que permita al equipo de marketing saber a qué cliente llamar primero, qué campaña aplicar y qué recursos invertir.  
> **Output:** `data/processed/final_customer_results.parquet` + 4 archivos CSV listos para Power BI

---

## 📋 Tabla de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [⚙️ Configuración](#1) | Setup, carga del modelo y datos |
| 2 | [🔴 Segmentación por Riesgo](#2) | Muy Alto / Medio / Estable |
| 3 | [💎 Segmentación RFM](#3) | VIP / Leales / En Riesgo |
| 4 | [📊 Distribución de Segmentos](#4) | Visualización de la composición del portafolio |
| 5 | [🧑‍💼 Perfiles Estadísticos](#5) | Métricas clave por segmento |
| 6 | [💰 Matriz Valor × Riesgo](#6) | El gráfico más importante para ejecutivos |
| 7 | [🚨 Lista de Acción Prioritaria](#7) | Top N clientes a contactar hoy |
| 8 | [📁 Exportar Resultados](#8) | CSV para Power BI + Parquet para pipeline |
| 9 | [✅ Conclusiones](#9) | Resumen ejecutivo |

---
## ⚙️ 1. Configuración <a id='1'></a>

In [ ]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yaml

plt.style.use('seaborn-v0_8-whitegrid')
COLORS     = {'primary':'#2d6a4f','secondary':'#52b788','accent':'#1a759f','warning':'#e76f51','danger':'#d62828','purple':'#6a0dad'}
RISK_COLORS = {'Riesgo Muy Alto': '#d62828', 'Riesgo Medio': '#e76f51', 'Estable (Riesgo Bajo)': '#2d6a4f'}
LEVEL_COLORS = {'VIP / Champion': '#6a0dad', 'Leales / Prometedores': '#1a759f', 'En Riesgo / Perdidos': '#d62828'}

with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

for d in ['data/processed', 'data/exports', 'reports/figures/eda']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Entorno configurado')

In [ ]:
from src.modeling.registry import ModelRegistry
from src.features.scaler import FeatureScaler

registry = ModelRegistry()
model, champion_info = registry.load_champion()
FEATURES = champion_info.get('features', cfg['features']['rfm'])
MODEL_NAME = champion_info.get('model_name', 'Champion')

rfm = pd.read_parquet('data/processed/rfm_with_churn.parquet')
features_available = [f for f in FEATURES if f in rfm.columns]

scaler_path = Path('models/champion/scaler.pkl')
if scaler_path.exists():
    scaler = FeatureScaler.load(scaler_path, features=features_available)
    rfm_scaled = scaler.transform(rfm)
else:
    rfm_scaled = rfm.copy()

print(f'✅ Modelo: {MODEL_NAME} | Clientes: {len(rfm):,}')

---
## 🔴 2. Segmentación por Riesgo <a id='2'></a>

> El `RiskSegmenter` usa las **probabilidades** del modelo para categorizar a cada cliente:

| Segmento | Probabilidad de Churn | Acción recomendada |
|----------|----------------------|--------------------|
| 🔴 **Riesgo Muy Alto** | ≥ 70% | Contacto inmediato, descuento de reactivación |
| 🟡 **Riesgo Medio** | 40% – 70% | Newsletter personalizada, puntos dobles |
| 🟢 **Estable** | < 40% | Programa de fidelización preventiva |

In [ ]:
from src.segmentation.risk_segmenter import RiskSegmenter

churn_cfg = cfg['churn']
risk_seg = RiskSegmenter(
    features=features_available,
    high_threshold=churn_cfg['high_risk_threshold'],
    medium_threshold=churn_cfg['medium_risk_threshold']
)
df_risk = risk_seg.transform(rfm_scaled, model)

print('=== Distribución por Segmento de Riesgo ===')
risk_dist = df_risk['RiskSegment'].value_counts()
for seg, count in risk_dist.items():
    icon = '🔴' if 'Muy Alto' in seg else '🟡' if 'Medio' in seg else '🟢'
    print(f'  {icon} {seg:<30} {count:>5,} clientes ({count/len(df_risk)*100:.1f}%)')

---
## 💎 3. Segmentación RFM <a id='3'></a>

> El `RFMSegmenter` asigna scores por cuartil (1-4) a cada métrica RFM y los combina:

| Nivel | RFM Score | Descripción |
|-------|-----------|-------------|
| 💎 **VIP / Champion** | ≥ 9 | Mejores clientes — compran reciente, frecuente y alto valor |
| 🔵 **Leales / Prometedores** | 6 – 8 | Clientes regulares con potencial de crecimiento |
| 🔴 **En Riesgo / Perdidos** | < 6 | Baja actividad histórica — difíciles de recuperar |

In [ ]:
from src.segmentation.rfm_segmenter import RFMSegmenter

seg_cfg = cfg['segmentation']
rfm_seg = RFMSegmenter(
    vip_threshold=seg_cfg['vip_rfm_threshold'],
    loyal_threshold=seg_cfg['loyal_rfm_threshold']
)
df_final = rfm_seg.transform(df_risk)

# Restaurar Monetary original (sin escalar)
if 'Monetary' in rfm.columns:
    df_final = df_final.copy()
    df_final['Monetary'] = rfm['Monetary'].values

print('=== Distribución por Nivel de Cliente (RFM) ===')
level_dist = df_final['CustomerLevel'].value_counts()
for level, count in level_dist.items():
    icon = '💎' if 'VIP' in level else '🔵' if 'Leales' in level else '🔴'
    print(f'  {icon} {level:<30} {count:>5,} clientes ({count/len(df_final)*100:.1f}%)')

---
## 📊 4. Distribución Visual de Segmentos <a id='4'></a>

> Visualización de la composición del portafolio de clientes por ambas dimensiones de segmentación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Composición del Portafolio de Clientes', fontsize=13, fontweight='bold')

# Riesgo
risk_counts = df_final['RiskSegment'].value_counts()
rc = [RISK_COLORS.get(r,'gray') for r in risk_counts.index]
w, t, pt = axes[0].pie(risk_counts.values, labels=risk_counts.index, colors=rc,
                        autopct='%1.1f%%', startangle=90,
                        wedgeprops={'edgecolor':'white','linewidth':2.5})
for p in pt: p.set_fontsize(10); p.set_fontweight('bold')
axes[0].set_title(f'Por Segmento de Riesgo (n={len(df_final):,})', fontweight='bold')

# Nivel RFM
level_counts = df_final['CustomerLevel'].value_counts()
lc = [LEVEL_COLORS.get(l,'gray') for l in level_counts.index]
w2, t2, pt2 = axes[1].pie(level_counts.values, labels=level_counts.index, colors=lc,
                           autopct='%1.1f%%', startangle=90,
                           wedgeprops={'edgecolor':'white','linewidth':2.5})
for p in pt2: p.set_fontsize(10); p.set_fontweight('bold')
axes[1].set_title(f'Por Nivel de Cliente RFM (n={len(df_final):,})', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/eda/segmentation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🧑‍💼 5. Perfiles Estadísticos por Segmento <a id='5'></a>

> Los perfiles estadísticos **validan** que los segmentos son coherentes con el dominio de negocio:  
> los clientes de Riesgo Muy Alto deben tener mayor Recency y menor Monetary que los estables.

In [ ]:
from src.segmentation.customer_profiler import CustomerProfiler

profiler = CustomerProfiler()

print('=== Perfil por Segmento de Riesgo ===')
profile_risk = profiler.profile_by_risk(df_final)
print(profile_risk.to_string())

print('\n=== Perfil por Nivel de Cliente RFM ===')
profile_level = profiler.profile_by_level(df_final)
print(profile_level.to_string())

In [ ]:
print('\n=== Exposición Monetaria por Segmento ===')
exposure = profiler.exposure_summary(df_final)
print(exposure.to_string(index=False))

total_exposed = df_final.loc[df_final['RiskSegment']!='Estable (Riesgo Bajo)', 'Monetary'].sum()
print(f'\n💰 Exposición monetaria total en riesgo : £{total_exposed:,.0f}')

---
## 💰 6. Matriz Valor × Riesgo <a id='6'></a>

> **El gráfico más poderoso para ejecutivos.** Cada punto es un cliente.  
> - **Eje X:** Probabilidad de churn (riesgo)  
> - **Eje Y:** Valor monetario histórico (£)  
> - **Zona crítica (arriba-derecha):** Alto valor + Alto riesgo → prioridad máxima

In [ ]:
sample = df_final.sample(min(1500, len(df_final)), random_state=42)
monetary_clip = rfm['Monetary'].quantile(0.99)
sample_monetary = rfm['Monetary'].loc[sample.index].clip(upper=monetary_clip)

fig, ax = plt.subplots(figsize=(13, 7))
for segment, color in RISK_COLORS.items():
    mask = sample['RiskSegment'] == segment
    ax.scatter(sample.loc[mask,'ChurnProbability'], sample_monetary[mask],
               c=color, alpha=0.45, s=18, label=segment, zorder=2)

ax.axvline(churn_cfg['medium_risk_threshold'], color='#e76f51', lw=1.5, ls='--', alpha=0.7,
           label=f'Umbral Medio ({int(churn_cfg["medium_risk_threshold"]*100)}%)')
ax.axvline(churn_cfg['high_risk_threshold'], color='#d62828', lw=2, ls='--', alpha=0.8,
           label=f'Umbral Alto ({int(churn_cfg["high_risk_threshold"]*100)}%)')

# Zona crítica
ax.axhspan(monetary_clip*0.5, monetary_clip, xmin=churn_cfg['high_risk_threshold'], alpha=0.05, color='red')
ax.annotate('⚠️ ZONA CRÍTICA\nAlto Valor + Alto Riesgo',
            xy=(0.85, monetary_clip*0.65), fontsize=9, color='#d62828', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='#d62828', alpha=0.85))

ax.set_xlabel('Probabilidad de Churn', fontsize=12)
ax.set_ylabel('Valor Monetario Histórico (£)', fontsize=12)
ax.set_title('💰 Matriz de Valor × Riesgo — Portafolio Completo de Clientes',
             fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'£{v:,.0f}'))
ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('reports/figures/eda/value_risk_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: reports/figures/eda/value_risk_matrix.png')

---
## 🚨 7. Lista de Acción Prioritaria <a id='7'></a>

> El **Priority Score** combina el valor del cliente con su probabilidad de churn:  
> `PriorityScore = Monetary × ChurnProbability`  
> → Maximiza el ROI de las campañas de retención (mayor impacto por esfuerzo invertido)

In [ ]:
top_priority = profiler.top_at_risk_customers(df_final, n=20)

cols_show = ['Customer ID','Monetary','ChurnProbability','RiskSegment','CustomerLevel','PriorityScore']
cols_available = [c for c in cols_show if c in top_priority.columns]

print('=== 🚨 TOP 20 CLIENTES PRIORITARIOS PARA RETENCIÓN ===')
print(top_priority[cols_available].round(3).to_string(index=False))

In [ ]:
# Heatmap RFM × Riesgo
rfm_pivot = df_final.pivot_table(
    values='ChurnProbability', index='CustomerLevel', columns='RiskSegment', aggfunc='mean'
).fillna(0)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(rfm_pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax,
            linewidths=1.5, vmin=0, vmax=1, cbar_kws={'shrink':0.8},
            annot_kws={'size':11, 'weight':'bold'})
ax.set_title('🔥 Probabilidad Media de Churn\nNivel RFM × Segmento de Riesgo', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda/rfm_risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📁 8. Exportar Resultados <a id='8'></a>

> Los 4 archivos CSV generados alimentan directamente el dashboard de Power BI.

| Archivo | Contenido | Uso en Power BI |
|---------|-----------|----------------|
| `customer_churn_results.csv` | Todos los clientes con predicciones | Tabla principal |
| `vip_at_risk.csv` | Solo VIPs en riesgo | Tarjeta de alerta VIP |
| `top_priority_customers.csv` | Top 50 por Priority Score | Lista de acción |
| `executive_summary.csv` | KPIs agregados | Tarjetas de KPI |

In [ ]:
# Guardar resultado final
df_final.to_parquet('data/processed/final_customer_results.parquet', index=False)

# Exportar CSVs
from src.export.csv_exporter import CSVExporter
csv_exporter = CSVExporter(exports_dir='data/exports')
exported = csv_exporter.export_all(df_final)

print('✅ Archivos exportados:')
for name, path in exported.items():
    size = Path(path).stat().st_size / 1024 if Path(path).exists() else 0
    print(f'   {name}: {path} ({size:.1f} KB)')

---
## ✅ 9. Conclusiones <a id='9'></a>

In [ ]:
high_risk_n   = (df_final['RiskSegment']=='Riesgo Muy Alto').sum()
vip_risk_n    = len(df_final[(df_final.get('CustomerLevel','')=='VIP / Champion') &
                              (df_final['RiskSegment'].isin(['Riesgo Muy Alto','Riesgo Medio']))])
exposure      = df_final.loc[df_final['RiskSegment']=='Riesgo Muy Alto','Monetary'].sum()

print('╔══════════════════════════════════════════════════════╗')
print('║    📊 RESUMEN DE SEGMENTACIÓN                       ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'  Total clientes      : {len(df_final):,}')
print(f'  🔴 Riesgo Muy Alto  : {high_risk_n:,} ({high_risk_n/len(df_final)*100:.1f}%)')
print(f'  💎 VIPs en riesgo   : {vip_risk_n}')
print(f'  💰 Exposición £     : £{exposure:,.0f}')
print('╠══════════════════════════════════════════════════════╣')
print('  📁 Outputs:')
print('  • data/processed/final_customer_results.parquet')
for name in ['customer_churn_results', 'vip_at_risk', 'top_priority_customers']:
    print(f'  • data/exports/{name}.csv')
print('  → Siguiente (último): 07_executive_summary.ipynb')
print('╚══════════════════════════════════════════════════════╝')

<div style="background: #d8f3dc; border-left: 5px solid #2d6a4f; padding: 15px 20px; border-radius: 6px; margin-top: 20px;">
  <strong>✅ Segmentación completada.</strong> Los clientes han sido clasificados en segmentos accionables con perfiles estadísticos validados.<br><br>
  Los archivos CSV están listos para conectar con <strong>Power BI</strong> (ver <code>docs/powerbi_setup_guide.md</code>).<br>
  Continúa en <strong>07_executive_summary.ipynb</strong> para el dashboard final dirigido a stakeholders.
</div>